In [1]:
import os
import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.v2 as v2
from torchvision.transforms import InterpolationMode
import torchvision.models as models



In [2]:
torch.manual_seed(3407)
torch.cuda.manual_seed(3407)
cudnn.deterministic = True
cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"
train_dir = "/kaggle/input/cassava-leaf-disease-classification/train_images/"
train_csv_path = "/kaggle/input/cassava-leaf-disease-classification/train.csv"

img_size = 384
batch_size = 32
num_workers = 4
num_classes = 5
epochs = 5  # increased a few epochs for better accuracy



Using device: cuda


In [3]:
model = models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:01<00:00, 26.4MB/s]


In [4]:
class CassavaDataset(Dataset):
    """Dataset that works for both training (with labels) and testing (labels optional)."""

    def __init__(self, img_dir, df=None, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        if df is not None:
            self.filenames = df["image_id"].tolist()
            self.labels = df["label"].tolist()
        else:
            self.filenames = [
                f
                for f in sorted(os.listdir(img_dir))
                if os.path.isfile(os.path.join(img_dir, f))
            ]
            self.labels = None

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        path = os.path.join(self.img_dir, fname)
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.labels is None:
            return img, fname
        else:
            label = self.labels[idx]
            return img, label




In [5]:
train_transform = v2.Compose(
    [
        v2.RandomResizedCrop((img_size, img_size), scale=(0.8, 1.0)),
        v2.RandomHorizontalFlip(),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

test_transform = v2.Compose(
    [
        v2.Resize((img_size, img_size), interpolation=InterpolationMode.BICUBIC),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_df = pd.read_csv(train_csv_path)
train_dataset = CassavaDataset(train_dir, df=train_df, transform=train_transform)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

test_dataset = CassavaDataset(test_dir, transform=test_transform)  # fixed typo
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)



In [6]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = torch.tensor(
            labels, dtype=torch.long, device=device
        )  # ensure correct type

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}")



/tmp/ipykernel_55/1620291100.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(


Epoch [1/5] - Loss: 0.5264
Epoch [2/5] - Loss: 0.3883
Epoch [3/5] - Loss: 0.3393
Epoch [4/5] - Loss: 0.2962
Epoch [5/5] - Loss: 0.2562


In [7]:
model.eval()
all_names = []
all_preds = []

with torch.no_grad():
    for imgs, filenames in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        preds = torch.argmax(logits, dim=1)
        all_names.extend(filenames)
        all_preds.extend(preds.cpu().tolist())

assert len(all_names) == len(all_preds), "Mismatch in prediction count"

submission = pd.DataFrame({"image_id": all_names, "label": all_preds})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path} with {len(submission)} rows")

Submission written to submission.csv with 2676 rows
